In [ ]:
import os
from pathlib import Path
import logging
import shutil
from datetime import datetime
import pandas as pd
from pyspark.sql import functions as F

import teehr
from teehr.evaluation.evaluation import RemoteReadOnlyEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.nwm.nwm_points import nwm_to_parquet
from teehr.fetching.utils import create_periods_based_on_chunksize, get_period_start_end_times
from teehr.fetching.utils import (
    format_nwm_variable_name,
    format_nwm_configuration_metadata
)
from teehr.utils.utils import remove_dir_if_exists
from teehr.fetching.const import NWM_VARIABLE_MAPPER

import dask
dask.config.set({"distributed.dashboard.link": "{JUPYTERHUB_SERVICE_PREFIX}proxy/{port}/status"})

logger = logging.getLogger()

In [ ]:
LOCATION_ID_PREFIX = "nwm30"
OCONUS_STATE_NAMES = [
    'Northern Mariana Islands', 'Alaska', 'Hawaii', 'Guam',
    'American Samoa', 'Puerto Rico', 'Virgin Islands'
]
nwm_version = "nwm30"

nwm_configuration = "medium_range_mem1"
output_type = "channel_rt_1"
variable_name = "streamflow"

- NWM v3.0: 2023-09-19 - present
- Earliest reference time in our secondary table: 2025-11-10 22:00:00

Create a local Evaluation to store the cached parquet files

In [ ]:
%%time
dir_path =  "/data/playground/slamont/teehr/warehouse/loading/backfill_nwm/local-evaluation-medium"

spark = create_spark_session()

ev = teehr.LocalReadWriteEvaluation(
    spark=spark,
    dir_path=dir_path,
    create_dir=True
)

Get the CONUS NWM IDs (these were pre-prepared from the warehouse using the query in fetch_nwm30_shortrange_streamflow.ipynb)

In [ ]:
df_nwm_ids = pd.read_csv("/data/playground/slamont/teehr/warehouse/loading/backfill_nwm/nwm_conus_ids.csv")
stripped_ids = df_nwm_ids["nwm_id"].tolist()

In [ ]:
ev_variable_name = format_nwm_variable_name(variable_name)
ev_config = format_nwm_configuration_metadata(
    nwm_config_name=nwm_configuration,
    nwm_version=nwm_version
)
nwm_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "nwm"
)
kerchunk_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "kerchunk"
)

Last file: 20240214T06F081_20240215T06F240.parquet

In [ ]:
start_date = datetime(2024, 2, 17, 0)  # 2023-09-19 (start of nwm v3.0)
end_date = datetime(2025, 11, 10, 0)   # 2025-11-10 22:00 (the earliest non-null reference_time in secondary)
chunk_by = "week"  # Describes the chunk size between start/end date

960 timesteps per day (4 * 240), 6720 (960 * 7) timesteps per week

In [ ]:
# Start up a local Dask cluster
from dask.distributed import Client
client = Client(n_workers=12)  # only using 4 workers by default?
client

Notes:
- `chunk_by` governs the total number of files/timesteps to fetch at once.

Within the total number of timesteps, processing is done in chunks governed by:
- `process_by_z_hour` - chunks by reference time
- `stepsize` - if above is False, an integer number of timesteps

In [ ]:
%%time
periods = create_periods_based_on_chunksize(
    start_date=start_date,
    end_date=end_date,
    chunk_by=chunk_by
)

for i, period in enumerate(periods):

    # if i <= 2:  # already processed this batch
    #     continue 
    
    if period is not None:
        dts = get_period_start_end_times(
            period=period,
            start_date=start_date,
            end_date=end_date
        )
    else:
        dts = {"start_dt": start_date, "end_dt": end_date}

    logger.info("Fetching NWM data and writing to cache")
    nwm_to_parquet(
        configuration=nwm_configuration,
        output_type=output_type,
        variable_name=variable_name,
        start_date=dts["start_dt"],
        end_date=dts["end_dt"],
        location_ids=stripped_ids,
        json_dir=kerchunk_cache_dir,
        output_parquet_dir=Path(
            nwm_cache_dir,
            ev_config["name"],
            ev_variable_name
        ),
        nwm_version=nwm_version,
        variable_mapper=NWM_VARIABLE_MAPPER,
        kerchunk_method="auto",
        process_by_z_hour=False,
        stepsize=1120,
        overwrite_output=True
    )    

#### Let's see what we've got

In [ ]:
parquet_cache_dir = Path(
    ev.cache_dir,
    "fetching",
    "nwm",
    ev_config["name"],
    ev_variable_name    
)
parquet_cache_dir

In [ ]:
cached_sdf = ev.spark.read.parquet(parquet_cache_dir.as_posix())

In [ ]:
cached_sdf.show(n=6, truncate=False)

In [ ]:
cached_sdf.count()

In [ ]:
cached_sdf.select(F.min("value_time")).show(), cached_sdf.select(F.max("value_time")).show()